In [16]:
def load_small_sample():
    sample_text = ""
    filename = "hi_part_1.txt"
    
    with open(filename, "r", encoding="utf-8", errors="ignore") as f:
        for j, line in enumerate(f):
            sample_text += line
            if j > 2000:   # read only first ~2000 lines
                break
    
    return sample_text


raw_text = load_small_sample()
print("Sample length:", len(raw_text))
print(raw_text[:500])

Sample length: 613270
शारदा पारा के मिलन चैiक से आज महापौर देवेन्द्र यादव, पार्षद छोटे लाल चैधरी के साथ वार्ड का भ्रमण करते हुए वार्ड की पेयजल, स्वच्छता सफाई, विकास कार्यों का निरीक्षण किया।
सूचना का अधिकार - विभाग द्वारा तैयार 17 column सम्बंधित पंजी ,कार्यालय नगर पालिक निगम भिलाई जोन-06 रिसाली
आज सम्पत्तिकर अधिकारी एच.के. चन्द्राकर ने महाप्रबंधक भारत संचार निगम लिमिटेड दुर्ग को सम्पत्तिकर की राशि कुर्कीवारण्ट जारी कर दिया, कुर्कीवारण्ट के साथ अधिभार सहित राशि वसुल किये जाने की नोटिस तामिल की गई है।
राष्ट्रपति, राज्


In [17]:
import tiktoken
print("tiktoken installed successfully")

tiktoken installed successfully


In [18]:
tokenizer = tiktoken.get_encoding("gpt2")

encoded = tokenizer.encode(raw_text)
print("Encoded tokens:", len(encoded))

Encoded tokens: 867612


In [19]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, token_ids, max_length=64, stride=32):
        self.input_ids = []
        self.target_ids = []

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

dataset = GPTDataset(encoded, max_length=64, stride=32)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

batch = next(iter(dataloader))
print("Input batch shape:", batch[0].shape)
print("Target batch shape:", batch[1].shape)

Input batch shape: torch.Size([8, 64])
Target batch shape: torch.Size([8, 64])


In [20]:

vocab_size = tokenizer.n_vocab
embedding_dim = 128

embedding = torch.nn.Embedding(vocab_size, embedding_dim)
sample_input = batch[0]
embedded_output = embedding(sample_input)

print("Embedded output shape:", embedded_output.shape)

Embedded output shape: torch.Size([8, 64, 128])


In [21]:

batch = next(iter(dataloader))
inputs, targets = batch

print("INPUT TEXT:")
print(tokenizer.decode(inputs[0].tolist()))

print("\nTARGET TEXT:")
print(tokenizer.decode(targets[0].tolist()))

INPUT TEXT:
� और वह सातवें नंबर पर फिसल गया है। विश्व क�

TARGET TEXT:
 और वह सातवें नंबर पर फिसल गया है। विश्व कप


In [22]:

input_ids = inputs[0]
target_ids = targets[0]

for i in range(5):
    print(
        tokenizer.decode([input_ids[i].item()]),
        "→",
        tokenizer.decode([target_ids[i].item()])
    )

� →  �
 � → �
� → �
� → �
� →  �


In [23]:
for i in range(5):
    print(input_ids[i].item(), "→", target_ids[i].item())

230 → 28225
28225 → 242
242 → 11976
11976 → 108
108 → 28225


In [24]:
for i in range(10, 16):
    print(
        tokenizer.decode(input_ids[:i].tolist()),
        "→",
        tokenizer.decode(target_ids[:i].tolist())
    )

� और वह � →  और वह स
� और वह स →  और वह सा
� और वह सा →  और वह सा�
� और वह सा� →  और वह सात
� और वह सात →  और वह सात�
� और वह सात� →  और वह सातव


In [25]:
print("Embedding weight shape:", embedding.weight.shape)
print("Sample input shape:", inputs.shape)
print("Embedded output shape:", embedded_output.shape)

Embedding weight shape: torch.Size([50257, 128])
Sample input shape: torch.Size([8, 64])
Embedded output shape: torch.Size([8, 64, 128])


In [26]:
lm_head = torch.nn.Linear(embedding_dim, vocab_size)

logits = lm_head(embedded_output)

print("Logits shape:", logits.shape)


Logits shape: torch.Size([8, 64, 50257])
